## 01 - Prepare PoQuAD
Load a small PoQuAD slice, preview normalized documents/queries, and write `documents.jsonl` and `queries.jsonl` for downstream chunking.

In [1]:
# Robustly set repo root and import path
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for cand in [ROOT, *ROOT.parents]:
    if (cand / "src").exists():
        ROOT = cand
        break
else:
    raise RuntimeError("Could not locate repo root containing src/.")

# Add repo root so imports like 'src.data_loader' resolve
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print(f"Repo root: {ROOT}")

Repo root: /home/jula/Repositories/Chunking-Research


### Configure dataset and output
Pick the dataset, slice for speed, and choose the output folder under `data/processed/poquad/<split>/`.

In [2]:
# Configure dataset slice and output
dataset_name = "poquad"  # registered in src/data_loader/datasets
split = "train"  # small slice for demo speed
output_dir = ROOT / "data" / "processed" / dataset_name / 'example'

loader_kwargs = {
    "split": split,
    "revision": "refs/convert/parquet",
    "cache_dir": str(ROOT / "data" / "hf_cache"),
    "limit": None,
}
output_dir

PosixPath('/home/jula/Repositories/Chunking-Research/data/processed/poquad/example')

### Load and preview
Use the registered loader; it returns normalized documents and queries ready to save.

In [3]:
from src.data_loader import get_dataset_loader
from src.data_loader.core.schemas import save_document_records_jsonl, save_query_records_jsonl

loader = get_dataset_loader(dataset_name)
documents, queries = loader(**loader_kwargs)
print(f"Loaded {len(documents)} documents")
print(f"Loaded {len(queries)} queries")
print("First document:\n", documents[0].to_dict())
print("First query:\n", queries[0].to_dict())

Loaded 11539 documents
Loaded 46187 queries
First document:
 {'id': 'poquad-fbd9d2dab0c6', 'contents': 'Projekty konfederacji zaczęły się załamywać 5 sierpnia 1942. Ponownie wróciła kwestia monachijska, co uaktywniło się wymianą listów Ripka – Stroński. Natomiast 17 sierpnia 1942 doszło do spotkania E. Beneša i J. Masaryka z jednej a Wł. Sikorskiego i E. Raczyńskiego z drugiej strony. Polscy dyplomaci zaproponowali podpisanie układu konfederacyjnego. W następnym miesiącu, tj. 24 września, strona polska przesłała na ręce J. Masaryka projekt deklaracji o przyszłej konfederacji obu państw. Strona czechosłowacka projekt przyjęła, lecz już w listopadzie 1942 E. Beneš podważył ideę konfederacji. W zamian zaproponowano zawarcie układu sojuszniczego z Polską na 20 lat (formalnie nastąpiło to 20 listopada 1942).', 'metadata': {'title': 'Konfederacja polsko-czechosłowacka'}}
First query:
 {'id': 'q.1', 'contents': 'Co było powodem powrócenia konceptu porozumieniu monachijskiego?', 'relevant': ['

### Persist outputs
Write `documents.jsonl` and `queries.jsonl` to the chosen directory.

In [4]:
output_dir.mkdir(parents=True, exist_ok=True)
documents_path = output_dir / "documents.jsonl"
queries_path = output_dir / "queries.jsonl"

save_document_records_jsonl(documents, documents_path)
save_query_records_jsonl(queries, queries_path)
print(f"Saved documents -> {documents_path}")
print(f"Saved queries   -> {queries_path}")

Saved documents -> /home/jula/Repositories/Chunking-Research/data/processed/poquad/example/documents.jsonl
Saved queries   -> /home/jula/Repositories/Chunking-Research/data/processed/poquad/example/queries.jsonl


### Merge documents (optional)

In [5]:
import json

project_root = None
current_path = Path.cwd()
while current_path != current_path.parent:
    if (current_path / "src").exists():
        project_root = current_path
        break
    current_path = current_path.parent

if project_root and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    from src.merge_documents import DocumentMerger
except ImportError:
    from src.merge_documents import DocumentMerger


SEED = 14
MERGED_OUTPUT_DIR = output_dir.parent / f"merged_seed{SEED}" 

docs_dicts = [d.to_dict() for d in documents]
queries_dicts = [q.to_dict() for q in queries]

merger = DocumentMerger(seed=SEED)
merged_docs, new_queries = merger.merge_dataset(docs_dicts, queries_dicts)

MERGED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

merged_docs_path = MERGED_OUTPUT_DIR / "documents.jsonl"
merged_queries_path = MERGED_OUTPUT_DIR / "queries.jsonl"

def save_simple_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

save_simple_jsonl(merged_docs, merged_docs_path)
save_simple_jsonl(new_queries, merged_queries_path)

print(f"Saved to directory: {MERGED_OUTPUT_DIR}")
print(f"Total merged documents: {len(merged_docs)}")
print(f"Total remapped queries: {len(new_queries)}")

[Merger] Processing 11539 documents into 'merged_doc_001'...
Saved to directory: /home/jula/Repositories/Chunking-Research/data/processed/poquad/merged_seed14
Total merged documents: 1
Total remapped queries: 46187
